In [29]:
import numpy as np
from pathlib import Path
from itertools import combinations
from collections import defaultdict
from scipy.optimize import linear_sum_assignment

SCENE = 'ParkingLot1_004_eating1'
BASE  = Path(f'/iopsstor/scratch/cscs/tnanni/ghost_outputs/rich11_segmentation_test/{SCENE}')

def load_tracks(cam: str) -> dict:
    body_dir = BASE / cam / 'body_data'
    persons = {}
    for npz_path in sorted(body_dir.glob('person_*.npz')):
        pid = int(npz_path.stem.split('_')[1])
        with np.load(str(npz_path)) as d:
            if 'pred_keypoints_3d' not in d or 'frame_indices' not in d:
                continue
            entry = {
                'kpts3d': d['pred_keypoints_3d'].copy(),   # (T, 70, 3)
                'frames': d['frame_indices'].copy(),
            }
            # pred_cam_t: metric body root position in camera space (z = metric depth)
            for key in ('pred_cam_t', 'smplx_transl'):
                if key in d:
                    entry['pred_cam_t'] = d[key].copy()   # (T, 3)
                    break
            persons[pid] = entry
    return persons

all_tracks = {}
for cam_dir in sorted(BASE.iterdir()):
    if not (cam_dir / 'body_data').exists():
        continue
    tracks = load_tracks(cam_dir.name)
    if tracks:
        all_tracks[cam_dir.name] = tracks

cam_list = sorted(all_tracks.keys())
print(f'Loaded {len(all_tracks)} cameras:')
for cam, persons in sorted(all_tracks.items()):
    print(f'  {cam}:')
    for pid, d in sorted(persons.items()):
        n = len(d['frames'])
        std = float(d['kpts3d'].std(axis=0).mean()) if n > 0 else 0.0
        has_t = 'pred_cam_t' in d
        print(f'    P{pid}: {n} frames  joint_std={std:.4f} m  pred_cam_t={has_t}')

Loaded 8 cameras:
  cam_00:
    P1: 522 frames  joint_std=0.1249 m  pred_cam_t=True
  cam_01:
    P1: 522 frames  joint_std=0.1265 m  pred_cam_t=True
    P2: 522 frames  joint_std=0.0084 m  pred_cam_t=True
    P3: 522 frames  joint_std=0.1065 m  pred_cam_t=True
    P5: 522 frames  joint_std=0.0350 m  pred_cam_t=True
    P6: 244 frames  joint_std=0.0859 m  pred_cam_t=True
  cam_02:
    P1: 522 frames  joint_std=0.1291 m  pred_cam_t=True
  cam_03:
    P1: 522 frames  joint_std=0.1169 m  pred_cam_t=True
    P2: 522 frames  joint_std=0.0094 m  pred_cam_t=True
    P3: 522 frames  joint_std=0.0623 m  pred_cam_t=True
    P4: 522 frames  joint_std=0.0385 m  pred_cam_t=True
    P5: 320 frames  joint_std=0.0936 m  pred_cam_t=True
    P6: 117 frames  joint_std=0.0395 m  pred_cam_t=True
    P9: 473 frames  joint_std=0.0781 m  pred_cam_t=True
    P10: 194 frames  joint_std=0.1016 m  pred_cam_t=True
  cam_04:
    P1: 522 frames  joint_std=0.1262 m  pred_cam_t=True
    P4: 522 frames  joint_std=0.011

In [30]:
# ── DROID-SLAM: estimate cam_10 camera motion ───────────────────────────────────

import sys, argparse
from tqdm.auto import tqdm as tqdm_slam
DROID_ROOT = Path('/users/tnanni/ghost/DROID-SLAM')
sys.path.insert(0, str(DROID_ROOT))
sys.path.insert(0, str(DROID_ROOT / 'droid_slam'))
import torch
import cv2

DROID_WEIGHTS  = '/capstor/scratch/cscs/tnanni/ghost_checkpoints/droid.pth'
RICH_DATA_ROOT = Path('/tmp/rich_mount')
MOVING_CAMS    = {'cam_10'}

_FX, _FY = 1200.0, 1200.0
_CX, _CY =  720.0,  526.0


def slam_image_stream(frames_dir, fx, fy, cx, cy):
    img_files = sorted(frames_dir.glob('*.jpeg'))
    if not img_files:
        img_files = sorted(frames_dir.glob('*.jpg'))
    for t, imfile in enumerate(img_files):
        image = cv2.imread(str(imfile))
        h0, w0 = image.shape[:2]
        scale  = np.sqrt((384 * 512) / (h0 * w0))
        h1, w1 = int(h0 * scale), int(w0 * scale)
        image  = cv2.resize(image, (w1, h1))
        image  = image[:h1 - h1 % 8, :w1 - w1 % 8]
        image  = torch.as_tensor(image).permute(2, 0, 1)
        intr   = torch.tensor([fx * w1/w0, fy * h1/h0, cx * w1/w0, cy * h1/h0])
        yield t, image[None], intr


def run_droid_slam(frames_dir, fx, fy, cx, cy, weights):
    from droid import Droid
    torch.multiprocessing.set_start_method('spawn', force=True)

    args = argparse.Namespace(
        weights=weights, image_size=[240, 320], buffer=512,
        stereo=False, disable_vis=True, beta=0.3, filter_thresh=2.4,
        warmup=8, keyframe_thresh=4.0, frontend_thresh=16.0,
        frontend_window=25, frontend_radius=2, frontend_nms=1,
        backend_thresh=22.0, backend_radius=2, backend_nms=3, upsample=False,
    )

    img_files = sorted(frames_dir.glob('*.jpeg')) or sorted(frames_dir.glob('*.jpg'))
    first = cv2.imread(str(img_files[0]))
    H_orig, W_orig = first.shape[:2]

    stream = list(slam_image_stream(frames_dir, fx, fy, cx, cy))
    H_slam, W_slam = None, None
    droid = None
    for t, image, intrinsics in tqdm_slam(stream, desc=f'DROID-SLAM {frames_dir.name}'):
        if droid is None:
            H_slam, W_slam = image.shape[2], image.shape[3]
            args.image_size = [H_slam, W_slam]
            droid = Droid(args)
        droid.track(t, image, intrinsics=intrinsics)

    poses = droid.terminate(iter(stream))  # (N_frames, 7) cam-to-world, poses[i] = i-th file

    n_kf   = droid.video.counter.value
    disps  = droid.video.disps[:n_kf].cpu().numpy()
    tstamp = droid.video.tstamp[:n_kf].cpu().numpy().astype(int)

    del droid
    torch.cuda.empty_cache()
    return poses, disps, tstamp, (H_orig, W_orig), (H_slam, W_slam)


slam_poses  = {}
slam_disps  = {}
slam_tstamp = {}
slam_hw     = {}

for cam in MOVING_CAMS:
    if cam not in all_tracks:
        continue
    frames_dir = RICH_DATA_ROOT / SCENE / cam
    if not frames_dir.exists():
        frames_dir = RICH_DATA_ROOT / SCENE / cam / 'frames'
    poses, disps, tstamp, orig_hw, slam_hw_cam = run_droid_slam(
        frames_dir, _FX, _FY, _CX, _CY, DROID_WEIGHTS)
    slam_poses[cam]  = poses
    slam_disps[cam]  = disps
    slam_tstamp[cam] = tstamp
    slam_hw[cam]     = (orig_hw, slam_hw_cam)
    print(f'  → {len(poses)} poses, {len(tstamp)} keyframes')

In [31]:
# ── SLAM scale estimation ───────────────────────────────────────────────────────
# λ such that t_slam_units * λ = t_metric.
# poses[i] = i-th JPEG file (sorted). frame_offset = first global frame of this cam.
# slam_index = global_frame_index - frame_offset

def estimate_slam_scale(cam, fx, fy, cx, cy, disps, tstamp, frame_offset, orig_hw, tracks):
    H_orig, W_orig = orig_hw
    H_disp, W_disp = disps.shape[1], disps.shape[2]
    sx = W_disp / W_orig
    sy = H_disp / H_orig

    # tstamp[kf_i] = SLAM enumerate index → global frame = tstamp[kf_i] + frame_offset
    lambdas = []
    for pid, data in tracks[cam].items():
        cam_t = data.get('pred_cam_t')
        if cam_t is None:
            continue
        frame_to_idx = {int(f): i for i, f in enumerate(data['frames'])}

        for kf_i, t in enumerate(tstamp):
            global_frame = int(t) + frame_offset
            arr_idx = frame_to_idx.get(global_frame)
            if arr_idx is None:
                continue

            x, y, z = cam_t[arr_idx]
            if z < 0.5:
                continue

            px = int(round((fx * x / z + cx) * sx))
            py = int(round((fy * y / z + cy) * sy))
            if not (0 <= px < W_disp and 0 <= py < H_disp):
                continue

            d_inv = float(disps[kf_i, py, px])
            if d_inv < 1e-6:
                continue

            lambdas.append(z * d_inv)

    if not lambdas:
        print(f'{cam}: WARNING — no valid λ estimates, defaulting to 1.0')
        return 1.0
    lam = float(np.median(lambdas))
    print(f'{cam}: λ = {lam:.4f}  (median over {len(lambdas)} keyframe detections, '
          f'disp shape {disps.shape})')
    return lam


slam_scales = {}
for cam in MOVING_CAMS:
    if cam not in slam_disps:
        continue
    frame_offset = int(min(d['frames'].min() for d in all_tracks[cam].values()))
    orig_hw, _   = slam_hw[cam]
    slam_scales[cam] = estimate_slam_scale(
        cam, _FX, _FY, _CX, _CY,
        slam_disps[cam], slam_tstamp[cam], frame_offset, orig_hw,
        all_tracks,
    )

In [32]:
# ── Apply SLAM poses: express cam_10 keypoints in metric world frame ────────────
# Full camera-frame position of keypoint j: pred_cam_t[i] + kpts[i, j]
# World position: R @ (pred_cam_t[i] + kpts[i, j]) + λ * t_slam
#               = R @ kpts[i, j]  +  R @ pred_cam_t[i]  +  λ * t_slam

def _quat_to_rot(q):
    """q: [qx, qy, qz, qw] → (3, 3) rotation matrix."""
    qx, qy, qz, qw = q
    return np.array([
        [1 - 2*(qy**2 + qz**2),  2*(qx*qy - qz*qw),  2*(qx*qz + qy*qw)],
        [  2*(qx*qy + qz*qw),  1 - 2*(qx**2 + qz**2), 2*(qy*qz - qx*qw)],
        [  2*(qx*qz - qy*qw),    2*(qy*qz + qx*qw), 1 - 2*(qx**2 + qy**2)],
    ])


for cam, poses in slam_poses.items():
    lam = slam_scales.get(cam, 1.0)
    frame_offset = int(min(d['frames'].min() for d in all_tracks[cam].values()))
    n_bad = 0
    for pid, data in all_tracks[cam].items():
        kpts      = data['kpts3d']          # (T, 70, 3) root-centred
        pred_cam_t = data.get('pred_cam_t') # (T, 3) absolute camera-frame root
        frames    = data['frames']
        corrected = np.full_like(kpts, np.nan)
        for i, frame_idx in enumerate(frames):
            t = int(frame_idx) - frame_offset
            if t < 0 or t >= len(poses) or not np.isfinite(poses[t]).all():
                n_bad += 1
                continue
            R = _quat_to_rot(poses[t, 3:])
            root_cam = pred_cam_t[i] if pred_cam_t is not None else np.zeros(3)
            # world = R @ (root + joint_offset) + λ*t = R@joint_offset + R@root + λ*t
            corrected[i] = (R @ kpts[i].T).T + R @ root_cam + lam * poses[t, :3]
        data['kpts3d'] = corrected
    print(f'{cam}: frame_offset={frame_offset}, λ={lam:.4f}, {n_bad} frames with no pose')


In [33]:
# ── Core functions (shared by all algorithm cells) ──────────────────────────────

def affine_fit(src: np.ndarray, dst: np.ndarray):
    """Standard 12-DOF affine (3×3 A + b) via least squares. src, dst: (N, 3)."""
    X = np.concatenate([src, np.ones((len(src), 1), dtype=src.dtype)], axis=1)
    valid = np.isfinite(X).all(axis=1) & np.isfinite(dst).all(axis=1)
    if valid.sum() < 12:   # need at least 12 points for 12 DOF
        return np.eye(3), np.zeros(3)
    M, _, _, _ = np.linalg.lstsq(X[valid], dst[valid], rcond=None)
    return M[:3].T, M[3]

def apply_affine(A, t, pts):
    """Apply standard affine to (N, 3) points."""
    return (A @ pts.T).T + t

def apply_T(A, t_vec, kpts):
    """Apply standard affine to (T, J, 3) keypoints."""
    shape = kpts.shape
    return (A @ kpts.reshape(-1, 3).T).T.reshape(shape) + t_vec

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx

def cluster_cams(uf, node):
    """Return the set of camera names already in node's cluster."""
    root = uf.find(node)
    return {cam for cam, pid in uf.parent if uf.find((cam, pid)) == root}

In [34]:
# ── Geometric cross-view ReID ────────────────────────────────────────────────────
# Full algorithm: RANSAC → joint refinement → Hungarian → Union-Find

# ── Thresholds ──────────────────────────────────────────────────────────────────
MIN_OVERLAP       = 30    # min common frames for a valid fit
MAX_DELTA         = 0    # max |δ| to search (TEST ONLY — remove for production)
INLIER_THR        = 0.25  # m — RANSAC inlier threshold
RANSAC_ANCHOR_THR = 0.20  # m — static camera pair rejection threshold
RANSAC_ANCHOR_THR_MOVING = 0.35  # m — relaxed threshold for moving-cam pairs
MATCH_RMSE_THR    = 0.25  # m — final merge threshold (static pairs)
MATCH_RMSE_THR_MOVING    = 0.35  # m — relaxed merge threshold for moving-cam pairs
DELTA_SEARCH      = 2     # frames — neighbourhood searched in joint refinement
MIN_A_SINGULAR_VALUE = 0.3  # m — reject affines where A collapses (degenerate/static target)
MIN_ANCHOR_STD       = 0.05 # m — prefer dynamic anchors; static-only scenes fall back gracefully


# ── Algorithm helpers ───────────────────────────────────────────────────────────

def get_aligned_slices(data_b, data_a, delta):
    fb_to_idx = {int(f): i for i, f in enumerate(data_b['frames'])}
    fa_to_idx = {int(f): i for i, f in enumerate(data_a['frames'])}
    common = sorted(f for f in fb_to_idx if f + delta in fa_to_idx)
    if len(common) < MIN_OVERLAP:
        return None, None
    idx_b = [fb_to_idx[f]         for f in common]
    idx_a = [fa_to_idx[f + delta] for f in common]
    return data_b['kpts3d'][idx_b], data_a['kpts3d'][idx_a]


def fit_pair_delta(data_b, data_a, delta):
    src, dst = get_aligned_slices(data_b, data_a, delta)
    if src is None:
        return None
    A, t = affine_fit(src.reshape(-1, 3), dst.reshape(-1, 3))
    if np.linalg.svd(A, compute_uv=False).min() < MIN_A_SINGULAR_VALUE:
        return None  # degenerate affine — static target or insufficient motion
    pred = apply_affine(A, t, src.reshape(-1, 3))
    rmse = float(np.sqrt(((pred - dst.reshape(-1, 3)) ** 2).sum(1).mean()))
    if not np.isfinite(rmse):
        return None
    return A, t, rmse


def direct_rmse_at_delta(A, t_vec, data_b, data_a, delta):
    src, dst = get_aligned_slices(data_b, data_a, delta)
    if src is None:
        return float('inf')
    pred = apply_T(A, t_vec, src)
    result = float(np.sqrt(((pred - dst) ** 2).sum(-1).mean()))
    return result if np.isfinite(result) else float('inf')


def get_inlier_pairs(A, t_vec, delta, valid_b, valid_a, anchor_pid_b):
    inliers = []
    for pid_b, data_b in valid_b.items():
        if pid_b == anchor_pid_b:
            continue
        best_rmse, best_pid_a = min(
            (direct_rmse_at_delta(A, t_vec, data_b, data_a, delta), pid_a)
            for pid_a, data_a in valid_a.items()
        )
        if best_rmse < INLIER_THR:
            inliers.append((pid_b, best_pid_a, best_rmse))
    return inliers


def joint_refine(assignment, valid_b, valid_a, delta_init):
    delta     = delta_init
    A, t_vec  = np.eye(3), np.zeros(3)
    prev_rmse = float('inf')

    # Only use pairs where at least one track is dynamic for fitting the transform.
    # Pairs where both tracks are static bias the affine the same way a static anchor does.
    # Fall back to all pairs if everything is static (e.g., all-seated scene).
    fit_pairs = [
        (pb, pa) for pb, pa in assignment
        if (float(valid_b[pb]['kpts3d'].std(axis=0).mean()) >= MIN_ANCHOR_STD or
            float(valid_a[pa]['kpts3d'].std(axis=0).mean()) >= MIN_ANCHOR_STD)
    ]
    if not fit_pairs:
        fit_pairs = assignment

    for _ in range(20):
        all_src, all_dst = [], []
        for pid_b, pid_a in fit_pairs:
            src, dst = get_aligned_slices(valid_b[pid_b], valid_a[pid_a], delta)
            if src is None:
                continue
            all_src.append(src.reshape(-1, 3))
            all_dst.append(dst.reshape(-1, 3))
        if not all_src:
            break

        A, t_vec = affine_fit(np.concatenate(all_src), np.concatenate(all_dst))

        best_d_rmse, best_d = float('inf'), delta
        for d in range(delta - DELTA_SEARCH, delta + DELTA_SEARCH + 1):
            sl_pred, sl_dst = [], []
            for pid_b, pid_a in fit_pairs:
                src, dst = get_aligned_slices(valid_b[pid_b], valid_a[pid_a], d)
                if src is None:
                    continue
                sl_pred.append(apply_T(A, t_vec, src).reshape(-1, 3))
                sl_dst.append(dst.reshape(-1, 3))
            if not sl_pred:
                continue
            rmse = float(np.sqrt(
                ((np.concatenate(sl_pred) - np.concatenate(sl_dst)) ** 2).sum(1).mean()))
            if np.isfinite(rmse) and rmse < best_d_rmse:
                best_d_rmse, best_d = rmse, d

        if abs(prev_rmse - best_d_rmse) < 1e-5:
            break
        prev_rmse, delta = best_d_rmse, best_d

    return A, t_vec, delta, best_d_rmse


# ── Main loop ───────────────────────────────────────────────────────────────────

uf = UnionFind()
for cam in cam_list:
    for pid in all_tracks[cam]:
        uf.find((cam, pid))

for cam_a, cam_b in combinations(cam_list, 2):
    persons_a, persons_b = all_tracks[cam_a], all_tracks[cam_b]
    if len(persons_b) > len(persons_a):
        cam_a, cam_b = cam_b, cam_a
        persons_a, persons_b = persons_b, persons_a

    valid_b = dict(persons_b)
    valid_a = dict(persons_a)
    is_moving_pair = bool(MOVING_CAMS & {cam_a, cam_b})
    anchor_thr = RANSAC_ANCHOR_THR_MOVING if is_moving_pair else RANSAC_ANCHOR_THR
    merge_thr  = MATCH_RMSE_THR_MOVING    if is_moving_pair else MATCH_RMSE_THR

    # ── Step 1: RANSAC ───────────────────────────────────────────────────────────
    # Two-stage anchor selection: prefer dynamic tracks (joint_std >= MIN_ANCHOR_STD)
    # so the inter-camera transform is estimated from temporal motion, not just body shape.
    # Falls back to best-RMSE anchor if no dynamic pair exists (all-static scene).
    _empty = {'inliers': -1, 'rmse': float('inf'),
              'A': None, 't': None, 'delta': 0, 'pid_b': None, 'pid_a': None}
    best_dyn = {**_empty}
    best_any = {**_empty}

    for pid_b, data_b in valid_b.items():
        std_b = float(data_b['kpts3d'].std(axis=0).mean())
        T_b = len(data_b['frames'])
        for pid_a, data_a in valid_a.items():
            std_a = float(data_a['kpts3d'].std(axis=0).mean())
            T_a = len(data_a['frames'])
            delta_min = max(-(T_b - 1), -MAX_DELTA)
            delta_max = min(T_a - 1,     MAX_DELTA)
            bd = {'rmse': float('inf'), 'A': None, 't': None, 'delta': 0}
            for delta in range(delta_min, delta_max + 1):
                res = fit_pair_delta(data_b, data_a, delta)
                if res and res[2] < bd['rmse']:
                    bd = {'rmse': res[2], 'A': res[0], 't': res[1], 'delta': delta}
            if bd['A'] is None:
                continue
            n_in = len(get_inlier_pairs(bd['A'], bd['t'], bd['delta'], valid_b, valid_a, pid_b))
            candidate = {**bd, 'inliers': n_in, 'pid_b': pid_b, 'pid_a': pid_a}
            if n_in > best_any['inliers'] or (n_in == best_any['inliers'] and bd['rmse'] < best_any['rmse']):
                best_any = candidate
            if std_b >= MIN_ANCHOR_STD and std_a >= MIN_ANCHOR_STD:
                if n_in > best_dyn['inliers'] or (n_in == best_dyn['inliers'] and bd['rmse'] < best_dyn['rmse']):
                    best_dyn = candidate

    if best_dyn['A'] is not None:
        best = best_dyn
    else:
        best = best_any
        if best_any['A'] is not None:
            print(f'  [no dynamic anchor — falling back to static anchor for {cam_a} × {cam_b}]')

    if best['A'] is None or best['rmse'] > anchor_thr:
        print(f'\n{cam_a} × {cam_b}:  [RANSAC failed — best RMSE={best["rmse"]:.3f} m]')
        continue

    sv = np.linalg.svd(best['A'], compute_uv=False)
    print(f'\n{cam_a} × {cam_b}:  '
          f'anchor {cam_b}/P{best["pid_b"]} → {cam_a}/P{best["pid_a"]}  '
          f'δ={best["delta"]}  RMSE={best["rmse"]:.3f} m  inliers={best["inliers"]}  '
          f'sv=[{sv.min():.2f},{sv.max():.2f}]')

    # ── Step 2: joint refinement ─────────────────────────────────────────────────
    inlier_pairs    = get_inlier_pairs(
        best['A'], best['t'], best['delta'], valid_b, valid_a, best['pid_b'])
    init_assignment = [(best['pid_b'], best['pid_a'])] + \
                      [(pb, pa) for pb, pa, _ in inlier_pairs]

    A, t_vec, delta, ref_rmse = joint_refine(init_assignment, valid_b, valid_a, best['delta'])
    print(f'  → after joint refinement: δ={delta}  RMSE={ref_rmse:.3f} m'
          f'  (used {len(init_assignment)} pairs)')

    # ── Step 3: Hungarian ────────────────────────────────────────────────────────
    pids_b = sorted(valid_b.keys())
    pids_a = sorted(valid_a.keys())
    rmse_mat = np.full((len(pids_b), len(pids_a)), 1e6)
    for i, pb in enumerate(pids_b):
        for j, pa in enumerate(pids_a):
            src, dst = get_aligned_slices(valid_b[pb], valid_a[pa], delta)
            n_frames = 0 if src is None else len(src)
            rmse = direct_rmse_at_delta(A, t_vec, valid_b[pb], valid_a[pa], delta)
            rmse_mat[i, j] = rmse if np.isfinite(rmse) else 1e6
            print(f'    {cam_b}/P{pb} → {cam_a}/P{pa}  frames={n_frames}  RMSE={rmse:.3f}')

    rmse_mat = np.where(np.isfinite(rmse_mat), rmse_mat, 1e6)
    row_ind, col_ind = linear_sum_assignment(rmse_mat)

    # ── Step 4: Union-Find ───────────────────────────────────────────────────────
    for r, c in zip(row_ind, col_ind):
        pid_b, pid_a, rmse_val = pids_b[r], pids_a[c], rmse_mat[r, c]
        row = rmse_mat[r].copy()
        row[c] = 1e6
        alt_order = np.argsort(row)[:2]
        alts = ''.join(f'  alt{k+1}: P{pids_a[alt_order[k]]}={row[alt_order[k]]:.3f}'
                       for k in range(min(2, len(alt_order))) if row[alt_order[k]] < 1e5)
        if rmse_val < merge_thr:
            conflict = cluster_cams(uf, (cam_a, pid_a)) & cluster_cams(uf, (cam_b, pid_b))
            if conflict:
                print(f'  P{pid_b} → P{pid_a}  RMSE={rmse_val:.3f}{alts}  '
                      f'[rejected — same-camera conflict: {conflict}]')
            else:
                print(f'  P{pid_b} → P{pid_a}  RMSE={rmse_val:.3f}{alts}  [merged]')
                uf.union((cam_a, pid_a), (cam_b, pid_b))
        else:
            print(f'  P{pid_b} → P{pid_a}  RMSE={rmse_val:.3f}{alts}  [absent, skipped]')

# ── Cluster summary ─────────────────────────────────────────────────────────────
by_gid = defaultdict(list)
for cam in cam_list:
    for pid in all_tracks[cam]:
        by_gid[uf.find((cam, pid))].append((cam, pid))

print('\n=== Final Clusters ===')
for gid, (_, members) in enumerate(sorted(by_gid.items()), 1):
    print(f'  Person {gid}: {"  ".join(f"{c}/P{p}" for c, p in sorted(members))}')


cam_01 × cam_00:  anchor cam_00/P1 → cam_01/P1  δ=0  RMSE=0.094 m  inliers=0  sv=[0.91,1.02]
  → after joint refinement: δ=0  RMSE=0.094 m  (used 1 pairs)
    cam_00/P1 → cam_01/P1  frames=522  RMSE=0.094
    cam_00/P1 → cam_01/P2  frames=522  RMSE=0.576
    cam_00/P1 → cam_01/P3  frames=522  RMSE=0.428
    cam_00/P1 → cam_01/P5  frames=522  RMSE=0.510
    cam_00/P1 → cam_01/P6  frames=244  RMSE=0.469
  P1 → P1  RMSE=0.094  alt1: P3=0.428  alt2: P6=0.469  [merged]

cam_00 × cam_02:  anchor cam_02/P1 → cam_00/P1  δ=0  RMSE=0.089 m  inliers=0  sv=[0.77,1.03]
  → after joint refinement: δ=0  RMSE=0.089 m  (used 1 pairs)
    cam_02/P1 → cam_00/P1  frames=522  RMSE=0.089
  P1 → P1  RMSE=0.089  [merged]

cam_03 × cam_00:  anchor cam_00/P1 → cam_03/P1  δ=0  RMSE=0.124 m  inliers=0  sv=[0.85,1.15]
  → after joint refinement: δ=0  RMSE=0.124 m  (used 1 pairs)
    cam_00/P1 → cam_03/P1  frames=522  RMSE=0.124
    cam_00/P1 → cam_03/P2  frames=522  RMSE=0.570
    cam_00/P1 → cam_03/P3  frames=52

In [9]:

import plotly.graph_objects as go

# ── Inspect keypoints for any track ─────────────────────────────────────────────
INSPECT_CAM   = 'cam_04'
INSPECT_PID   = 3
INSPECT_FRAME = 0   # array index (not actual frame number)

MHR70_EDGES = [
    (13,11),(11,9),(14,12),(12,10),(9,10),
    (5,9),(6,10),(5,6),(69,5),(69,6),
    (5,7),(7,62),(6,8),(8,41),
    (0,1),(0,2),(1,2),(1,3),(2,4),(3,5),(4,6),
    (13,15),(13,16),(13,17),(14,18),(14,19),(14,20),
    (62,45),(45,44),(44,43),(43,42),(62,49),(49,48),(48,47),(47,46),
    (62,53),(53,52),(52,51),(51,50),(62,57),(57,56),(56,55),(55,54),
    (62,61),(61,60),(60,59),(59,58),
    (41,24),(24,23),(23,22),(22,21),(41,28),(28,27),(27,26),(26,25),
    (41,32),(32,31),(31,30),(30,29),(41,36),(36,35),(35,34),(34,33),
    (41,40),(40,39),(39,38),(38,37),
]

data   = all_tracks[INSPECT_CAM][INSPECT_PID]
kpts   = data['kpts3d']    # (T, 70, 3)
frames = data['frames']
T      = len(frames)

print(f'{INSPECT_CAM}/P{INSPECT_PID}: {T} frames  '
      f'[frame {int(frames[0])} – {int(frames[-1])}]')
print(f'  x: [{kpts[:,:,0].min():.3f},  {kpts[:,:,0].max():.3f}]  '
      f'y: [{kpts[:,:,1].min():.3f},  {kpts[:,:,1].max():.3f}]  '
      f'z: [{kpts[:,:,2].min():.3f},  {kpts[:,:,2].max():.3f}]')
print(f'  mean joint std across all frames: {kpts.std(axis=0).mean():.4f} m  '
      f'(full-body typically ~0.15–0.30 m)')

# skeleton plot — hip-centred
kp = kpts[INSPECT_FRAME]
root = (kp[9] + kp[10]) / 2
kp = kp - root

xs, ys, zs = [], [], []
for a, b in MHR70_EDGES:
    xs += [kp[a,0], kp[b,0], None]
    ys += [kp[a,1], kp[b,1], None]
    zs += [kp[a,2], kp[b,2], None]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=kp[:,0], y=kp[:,1], z=kp[:,2],
    mode='markers', marker=dict(size=4, color='steelblue'), name='joints'))
fig.add_trace(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='lines', line=dict(color='steelblue', width=3), showlegend=False))
fig.update_layout(
    title=f'{INSPECT_CAM}/P{INSPECT_PID} — frame idx {INSPECT_FRAME} '
          f'(actual frame {int(frames[INSPECT_FRAME])})  |  {T} frames total',
    scene=dict(aspectmode='data'),
    height=600, margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()


cam_04/P3: 139 frames  [frame 554 – 692]
  x: [-0.272,  0.306]  y: [-1.602,  -0.055]  z: [-0.467,  0.211]
  mean joint std across all frames: 0.0448 m  (full-body typically ~0.15–0.30 m)
